The following has been adapted from Hailo's DFC Tutorials 1 and 2 (Parsing and Optimizing with DFC). It was run from a docker container setup using Hailo AI SoftwareSuite


In [1]:
# General imports used throughout the tutorial
# file operations
import json
import os

import numpy as np
import tensorflow as tf
from IPython.display import SVG
from matplotlib import patches
from matplotlib import pyplot as plt
from PIL import Image
from tensorflow.python.eager.context import eager_mode

import torchvision as tv
import torch

import cv2

# import the hailo sdk client relevant classes
from hailo_sdk_client import ClientRunner, InferenceContext

%matplotlib inline

IMAGES_TO_VISUALIZE = 5

In [2]:
chosen_hw_arch = "hailo8"

The ONNX model below was created using the `ultralytics` yolo11n.pt pretrained model, which was finetuned on a customised visdrone dataset (single class) and exported using:
```model.export(format='onnx', opset=14)```
    
the onnx output model was copied to the docker container into /local/shared_with_docker/yolov11n_visdrone.onnx
    
 

In [3]:
onnx_model_name = "yolo11n_visdrone"
onnx_path = "/local/shared_with_docker/yolo11n_visdrone_2class.onnx"

The endnodes below were taken from the [yolov11n.yaml](https://github.com/hailo-ai/hailo_model_zoo/blob/master/hailo_model_zoo/cfg/networks/yolov11n.yaml) on the hailo_model_zoo github. The finetuned (on VisDroneCustomClass) onnx model was opened in netron to double check these end nodes
```
- /model.23/cv2.0/cv2.0.2/Conv
- /model.23/cv3.0/cv3.0.2/Conv
- /model.23/cv2.1/cv2.1.2/Conv
- /model.23/cv3.1/cv3.1.2/Conv
- /model.23/cv2.2/cv2.2.2/Conv
- /model.23/cv3.2/cv3.2.2/Conv
```

In [4]:
runner = ClientRunner(hw_arch=chosen_hw_arch)
hn, npz = runner.translate_onnx_model(
    onnx_path,
    onnx_model_name,
    start_node_names=["/model.0/conv/Conv"],
    end_node_names=["/model.23/cv2.0/cv2.0.2/Conv", 
                   "/model.23/cv3.0/cv3.0.2/Conv",
                   "/model.23/cv2.1/cv2.1.2/Conv",
                   "/model.23/cv3.1/cv3.1.2/Conv",
                   "/model.23/cv2.2/cv2.2.2/Conv",
                   "/model.23/cv3.2/cv3.2.2/Conv"],
    net_input_shapes={"/model.0/conv/Conv": [1, 3, 640, 640]},
)

[info] Translation started on ONNX model yolo11n_visdrone
[info] Restored ONNX model yolo11n_visdrone (completion time: 00:00:00.05)
[info] Extracted ONNXRuntime meta-data for Hailo model (completion time: 00:00:00.18)
[info] NMS structure of yolov8 (or equivalent architecture) was detected.
[info] In order to use HailoRT post-processing capabilities, these end node names should be used: /model.23/cv3.0/cv3.0.2/Conv /model.23/cv2.0/cv2.0.2/Conv /model.23/cv3.1/cv3.1.2/Conv /model.23/cv2.1/cv2.1.2/Conv /model.23/cv2.2/cv2.2.2/Conv /model.23/cv3.2/cv3.2.2/Conv.
[info] Start nodes mapped from original model: 'images': 'yolo11n_visdrone/input_layer1'.
[info] End nodes mapped from original model: '/model.23/cv2.0/cv2.0.2/Conv', '/model.23/cv3.0/cv3.0.2/Conv', '/model.23/cv2.1/cv2.1.2/Conv', '/model.23/cv3.1/cv3.1.2/Conv', '/model.23/cv2.2/cv2.2.2/Conv', '/model.23/cv3.2/cv3.2.2/Conv'.
[info] Translation completed on ONNX model yolo11n_visdrone (completion time: 00:00:00.82)


In [5]:
#save the parsed model
hailo_model_har_name = f"{onnx_model_name}_hailo_model_op14.har"
runner.save_har(hailo_model_har_name)

[info] Saved HAR to: /local/workspace/hailo_virtualenv/lib/python3.10/site-packages/hailo_tutorials/notebooks/yolo11n_visdrone_hailo_model_op14.har


## Model Optimization

In [6]:
def preproc(image, output_height=640, output_width=640):
    preprocess = tv.transforms.Compose([
        tv.transforms.Resize((output_height, output_width)),
    ])
    
    data = np.array(preprocess(image))
    
    return data

In [7]:
# quantization needs calibration data, use the training data for this
cal_images_path = "../data/visdrone/train/images/" # use training data for calib
cal_images_list = [img_name for img_name in os.listdir(cal_images_path) if os.path.splitext(img_name)[1] == ".jpg"]
dataset_sz = 4100 
calib_dataset = np.zeros((dataset_sz, 640, 640, 3))

for idx, img_name in enumerate(sorted(cal_images_list)):
    if idx==dataset_sz:
        break
    img = Image.open(os.path.join(cal_images_path, img_name)).convert('RGB')
    img_preproc = preproc(img)
    calib_dataset[idx, :, :, :] = img_preproc
    
print(f"Calibration dataset size: {calib_dataset.shape}")

Calibration dataset size: (4100, 640, 640, 3)


In [8]:
#load our parsed HAR from the Parsing Tutorial
assert os.path.isfile(hailo_model_har_name), "Please provide valid path for HAR file"
runner = ClientRunner(har=hailo_model_har_name)

In [9]:
# Now we will create a model script, that tells the compiler to add a normalization on the beginning
# of the model (that is why we didn't normalize the calibration set;
# Otherwise we would have to normalize it before using it)
alls =  """
normalization1 = normalization([0.0, 0.0, 0.0], [255.0, 255.0, 255.0])
change_output_activation(conv54, sigmoid)
change_output_activation(conv65, sigmoid)
change_output_activation(conv80, sigmoid)
nms_postprocess("../yolov11_nms_config_visdrone.json", meta_arch=yolov8, engine=cpu)

model_optimization_config(calibration, batch_size=16, calibset_size=4000)
model_optimization_flavor(optimization_level=3)
#post_quantization_optimization(dataset_size=4000)

allocator_param(width_splitter_defuse=disabled)
 """
#post_quantization_optimization(finetune, policy=enabled, learning_rate=0.0001, epochs=8, dataset_size=4000)



# Load the model script to ClientRunner so it will be considered on optimization
runner.load_model_script(alls)
#runner.optimize_full_precision()
runner.optimize(calib_dataset)

[info] Loading model script commands to yolo11n_visdrone from string
[info] Starting Model Optimization
[info] Model received quantization params from the hn
[info] MatmulDecompose skipped
[info] Starting Mixed Precision
[info] Model Optimization Algorithm Mixed Precision is done (completion time is 00:00:00.48)
[info] LayerNorm Decomposition skipped
[info] Starting Statistics Collector
[info] Using dataset with 4000 entries for calibration


Calibration: 100%|███████████████████████████████████████████████████████████████| 4000/4000 [00:51<00:00, 78.08entries/s]


[info] Model Optimization Algorithm Statistics Collector is done (completion time is 00:00:52.64)
[info] Starting Fix zp_comp Encoding
[info] Model Optimization Algorithm Fix zp_comp Encoding is done (completion time is 00:00:00.00)
[info] Starting Matmul Equalization
[info] Model Optimization Algorithm Matmul Equalization is done (completion time is 00:00:00.05)
[info] activation fitting started for yolo11n_visdrone/reduce_sum_softmax1/act_op
[info] Finetune encoding skipped
[info] Bias Correction skipped
[warning] Dataset is larger than dataset_size in Adaround. Increasing the algorithm dataset size might improve the results
[info] Starting Adaround
[info] The algorithm Adaround will use up to 4.61 GB of storage space
[info] Using dataset with 256 entries for Adaround
[info] Using dataset with 64 entries for bias correction


Adaround:   1%|▎                          | 1/100 [00:07<11:34,  7.01s/blocks, Layers=['yolo11n_visdrone/conv1_output_0']]

[warning] DALI is not installed, using tensorflow dataset for layer by layer train. Using DALI will improve train time significantly. To install it use: pip install --extra-index-url https://developer.download.nvidia.com/compute/redist nvidia-dali-cuda110 nvidia-dali-tf-plugin-cuda110
[warning] Dataset isn't shuffled without DALI. To remove this warning add the following model script command: `post_quantization_optimization(adaround, shuffle=False)`


...

Training:  45%|█▊  | 1146/2560 [00:05<00:05, 269.68batches/s, l2_loss: 0.0000 - round_loss: 0.4663 - annealing_b: 15.0957]


Adaround: 100%|█████████████| 100/100 [1:10:25<00:00, 42.26s/blocks, Layers=['yolo11n_visdrone/postprocess_output_layer']]


[info] Model Optimization Algorithm Adaround is done (completion time is 01:10:30.34)
[info] Quantization-Aware Fine-Tuning skipped
[info] Starting Layer Noise Analysis


Full Quant Analysis: 100%|█████████████████████████████████████████████████████████| 1/1 [01:54<00:00, 114.20s/iterations]


[info] Model Optimization Algorithm Layer Noise Analysis is done (completion time is 00:01:56.23)
[info] Model Optimization is done


In [10]:
model_name = "yolo11n_visdrone"
# fp_model_har_path = f"{model_name}_fp_opt_model_visdrone.har"
# runner.save_har(fp_model_har_path)

quant_model_har_path = f"{model_name}_visdrone2class_quantized_lvl3_opt.har"
runner.save_har(quant_model_har_path)

[info] Saved HAR to: /local/workspace/hailo_virtualenv/lib/python3.10/site-packages/hailo_tutorials/notebooks/yolo11n_visdrone_visdrone2class_quantized_lvl3_opt.har


In [11]:
# load the model
runner = ClientRunner(har=quant_model_har_path, hw_arch=chosen_hw_arch)

# fp_opt_model_har_path = 'yolo11n_visdrone_fp_opt_model_visdrone.har'
# runner = ClientRunner(har=fp_opt_model_har_path, hw_arch=chosen_hw_arch)

## Model Inference and Evaluation

In [12]:
# get a list of images (filename) and the corresponding image_id from annotations file

annotations_file = '/local/shared_with_docker/visdrone/annotations_VisDroneHumans_val.json'
images_path = "/local/shared_with_docker/VisDrone2019-DET-val/images"

with open(annotations_file, 'r') as f:
    val_gt = json.load(f)
    f.close()
    
images = val_gt['images']
image_list = [(x['file_name'],x['id']) for x in images] 

    


In [13]:
def run_inference(runner, sdk_type, input_data ):
    if sdk_type == 'quantized':
        with runner.infer_context(InferenceContext.SDK_QUANTIZED) as ctx:
            output = runner.infer(ctx, input_data)
    else:
        with runner.infer_context(InferenceContext.SDK_FP_OPTIMIZED) as ctx:
            output = runner.infer(ctx, input_data)
    return output  

def remove_zero_padding(output, imgid):
    # remove zero-padding and transpose detections into single array
    combined = np.empty((7, 0)) # 1 classid, xywh, score

    for i in range(output.shape[1]):
        print(f"processing class {i} for image {imgid}...")
        valid_mask = np.any(output[0,i,:,:] != 0, axis=(0))  # Check if any non-zero values exist in each column
        last_valid_indices = np.argmax(~valid_mask, axis=0) # First occurrence of zero padding

        dets = output[0, i, :, :last_valid_indices]
        class_col = np.full_like(dets[0,None], i) # add the classID into the array
        imgid_col = np.full_like(dets[0,None], imgid)
        dets = np.vstack((dets, class_col, imgid_col))
        combined = np.concatenate((combined, dets), axis=1)
        print(f"\tfound {dets.shape[1]} detections for class {i}...")

    dets = combined.T
    return dets


def swap_columns(detections):
    detections[:, [0, 1,2,3]] = detections[:, [1, 0, 3, 2]]
    return detections


def ltxy2xywh(xywh):
    xywh[:,0] = xywh[:,0] # x
    xywh[:,1] = xywh[:,1] # y
    xywh[:,2] = xywh[:,2] - xywh[:,0] # x2 - x1
    xywh[:,3] = xywh[:,3] - xywh[:,1] # y2 - y1

    return xywh

def rescale_bbox(detections, w, h):
    w_arr = np.full_like(detections[:,0], w)
    h_arr = np.full_like(detections[:,0], h)
    
    detections[:,0] = detections[:,0] * w_arr
    detections[:,1] = detections[:,1] * h_arr
    detections[:,2] = detections[:,2] * w_arr
    detections[:,3] = detections[:,3] * h_arr
    
    return detections


def get_image_metadata(images_dict, img_id):
    for img in images_dict:
        if img['id'] == img_id:
            w, h = img['width'], img['height']
            name = img["file_name"]

    return w,h, name


In [14]:
# create an np array for the validation dataset
dataset_sz = len(image_list)
val_dataset = np.zeros((dataset_sz, 640, 640, 3))
val_imageids = np.zeros((dataset_sz,1))
for idx, imagename_id in enumerate(image_list):
    imgname, imgid = imagename_id
    image_file = os.path.join(images_path, imgname)
    if idx==dataset_sz:
        break
    img = Image.open(image_file).convert('RGB')
    img_preproc = preproc(img)
    val_dataset[idx, :, :, :] = img_preproc
    val_imageids[idx] = imgid

In [15]:
def detections_batch_generation(dataset, dataset_id, images_dict):
    batch_dets = run_inference(runner=runner, sdk_type='quantized', input_data=dataset)
    all_dets = []
    for idx, output in enumerate(batch_dets):
        imgid = dataset_id[idx]
        
        # postprocessing steps
        # remove zero padding
        output = output.reshape(1,2,5,100) # expects a batch
        detections = remove_zero_padding(output, imgid)
        
        # convert to xywh
        xywh = np.full_like(detections, detections)
        xywh = swap_columns(xywh) # annoyingly, it is in yx, ont xy, what misery
        xywh = ltxy2xywh(xywh) # convert x2/y2 to w/h
        
        # rescale detections
        w,h, name = get_image_metadata(images_dict, imgid) # images 
        dets_scaled = rescale_bbox(xywh, w, h)
        
        all_dets.append(dets_scaled)
    return(all_dets)


def batch_detections_to_json(batch):
    class_remap = {0: 1, 1: 2}
    res = []
    annId = 1
    for detections in batch:
        for el in detections:
            x,y,w,h,conf,classid,imgid = el
            imgid = int(imgid)
            res.append({"image_id": imgid, "category_id": class_remap[classid], "bbox": [x,y,w,h], "score": conf, "id": annId, "segmentation": []}) 
            annId +=1
    return res


def save_json_file(filename, data):
    with open(outputfile, 'w') as outf:
        json.dump(data, outf)
        outf.close()

    print(f"Total of {len(processed_outputs)} annotations saved to {outputfile}")

In [16]:
batch_detections = detections_batch_generation(val_dataset, val_imageids, images)
processed_outputs = batch_detections_to_json(batch_detections)

outputfile = f"/local/shared_with_docker/visdrone/detections_VisDrone_val_cfg2.json" 
save_json_file(outputfile, processed_outputs)

[info] Using 1 GPU for inference


Inference: 536entries [00:42, 12.62entries/s]


processing class 0 for image [1.]...
	found 9 detections for class 0...
processing class 1 for image [1.]...
	found 0 detections for class 1...
processing class 0 for image [2.]...
	found 6 detections for class 0...
processing class 1 for image [2.]...
	found 17 detections for class 1...
processing class 0 for image [3.]...
	found 6 detections for class 0...
processing class 1 for image [3.]...
	found 5 detections for class 1...
processing class 0 for image [4.]...
	found 13 detections for class 0...
processing class 1 for image [4.]...
	found 29 detections for class 1...
processing class 0 for image [5.]...
	found 43 detections for class 0...
processing class 1 for image [5.]...
	found 2 detections for class 1...
processing class 0 for image [6.]...
	found 7 detections for class 0...
processing class 1 for image [6.]...
	found 1 detections for class 1...
processing class 0 for image [7.]...
	found 3 detections for class 0...
processing class 1 for image [7.]...
	found 2 detections for

processing class 0 for image [456.]...
	found 5 detections for class 0...
processing class 1 for image [456.]...
	found 13 detections for class 1...
processing class 0 for image [457.]...
	found 11 detections for class 0...
processing class 1 for image [457.]...
	found 8 detections for class 1...
processing class 0 for image [458.]...
	found 13 detections for class 0...
processing class 1 for image [458.]...
	found 20 detections for class 1...
processing class 0 for image [459.]...
	found 17 detections for class 0...
processing class 1 for image [459.]...
	found 13 detections for class 1...
processing class 0 for image [460.]...
	found 24 detections for class 0...
processing class 1 for image [460.]...
	found 8 detections for class 1...
processing class 0 for image [461.]...
	found 6 detections for class 0...
processing class 1 for image [461.]...
	found 4 detections for class 1...
processing class 0 for image [462.]...
	found 14 detections for class 0...
processing class 1 for image [

## Redundant functions and code
This was used for single image inference, e.g. for iterating over each image

In [17]:
# these functions are used for single image evaluation
# intented for use when iterating over the dataset
def get_input_data(image_dataset, image_id):
    assert image_id >0, f"Image id should be >0 as per coco formatting"

    imgname, imgid = image_dataset[image_id-1]
    imgfile = os.path.join(images_path, imgname)
    print(f"opening image at {imgfile}")
    img = Image.open(imgfile).convert('RGB')
    img_preproc = preproc(img)
    img_input = img_preproc.reshape(1,640,640,3)
    
    return img_input, imgid, img

def get_single_image(image_rootpath, image_file):
    imgfile = os.path.join(image_rootpath, imgname)
    print(f"opening image at {imgfile}")
    img = Image.open(imgfile).convert('RGB')
    img_preproc = preproc(img)
    img_input = img_preproc.reshape(1,640,640,3)
    
    return img_input, img

def detections_to_json(detections):
    class_remap = {0: 1, 1: 2}
    res = []
    annId = 1
    for el in detections:
        x,y,w,h,conf,classid,imgid = el
        imgid = int(imgid)
        res.append({"image_id": imgid, "category_id": class_remap[classid], "bbox": [x,y,w,h], "score": conf, "id": annId, "segmentation": []}) 
        annId +=1
    return res

def get_single_image(image_rootpath, image_file):
    imgfile = os.path.join(image_rootpath, image_file)
    print(f"opening image at {imgfile}")
    img = Image.open(imgfile).convert('RGB')
    img_preproc = preproc(img)
    img_input = img_preproc.reshape(1,640,640,3)
    
    return img_input, img

def generate_detections_file(image_list, image_dir):
    all_dets = []
    for idx, imgdata in enumerate(image_list):
        imgname, imgid = imgdata
        img_input, img = get_single_image(image_dir, imgname)
        
        output = run_inference(runner=runner, sdk_type='x', input_data=img_input)
        detections = remove_zero_padding(output, imgid)

        xywh = np.full_like(detections, detections)
        xywh = swap_columns(xywh) # annoyingly, it is in yx, ont xy, what misery
        xywh = ltxy2xywh(xywh) # convert x2/y2 to w/h


        w,h, name = get_image_metadata(images, imgid)

        dets_scaled = rescale_bbox(xywh, w, h)
        all_dets.append(dets_scaled)
    return(all_dets)
# full_data_dets = generate_detections_file(image_list, images_path)

    

# img_input, imgid, img = get_input_data(image_list, 1)
# output = run_inference(runner=runner, sdk_type='opt', input_data=img_input)
# detections = remove_zero_padding(output, imgid)

# xywh = np.full_like(detections, detections)
# xywh = swap_columns(xywh) # annoyingly, it is in yx, ont xy, what misery
# xywh = ltxy2xywh(xywh) # convert x2/y2 to w/h


# w,h, name = get_image_metadata(images, imgid)

# dets_scaled = rescale_bbox(xywh, w, h)

# processed_outputs = detections_to_json(dets_scaled)
# outputfile = f"/local/shared_with_docker/visdrone/detections_VisDrone_quant_dets.json" 
# with open(outputfile, 'w') as outf:
#     json.dump(processed_outputs, outf)
#     outf.close()
    
# print(f"Total of {len(processed_outputs)} annotations saved to {outputfile}")